In [4]:
import torch
import torch.nn as nn
import numpy as np
from collections import deque
import random
# ATT&CK技术映射
TECHNIQUES = ["扫描", "SQL注入", "恶意软件", "钓鱼", "C2", "Flooding"]

class ACNetwork(nn.Module):
    """Actor-Critic网络架构"""
    def __init__(self, input_dim, action_dim):
        super().__init__()
        
        # 共享特征提取层
        self.lstm = nn.LSTM(input_dim, 128, batch_first=True)
        self.fc_shared = nn.Linear(128 + 36, 256)  # 128 LSTM + 6x6图状态
        
        # Actor分支
        self.actor = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )
        
        # Critic分支
        self.critic = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        
    def forward(self, seq_input, graph_input):
        # 序列特征提取
        lstm_out, _ = self.lstm(seq_input)
        seq_feat = lstm_out[:, -1, :]
        
        # 图状态处理（展平）
        graph_feat = graph_input.view(-1, 36)
        
        # 特征融合
        joint_feat = torch.cat([seq_feat, graph_feat], dim=1)
        shared_feat = torch.relu(self.fc_shared(joint_feat))
        
        # 输出
        action_probs = torch.softmax(self.actor(shared_feat), dim=-1)
        state_value = self.critic(shared_feat)
        
        return action_probs, state_value

class ACAgent:
    def __init__(self, state_dim, action_dim):
        self.network = ACNetwork(state_dim, action_dim)
        self.optimizer = torch.optim.Adam(self.network.parameters(), lr=1e-4)
        
        # 经验池
        self.memory = deque(maxlen=10000)
        self.gamma = 0.99
        
    def get_action(self, seq_state, graph_state):
        seq_tensor = torch.FloatTensor(seq_state).unsqueeze(0)
        graph_tensor = torch.FloatTensor(graph_state).unsqueeze(0)
        
        with torch.no_grad():
            probs, _ = self.network(seq_tensor, graph_tensor)
        
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)
    
    def update(self):
        if len(self.memory) < 32:
            return
        
        # 从经验池采样
        batch = random.sample(self.memory, 32)
        states, actions, log_probs, rewards, next_states, dones = zip(*batch)
        
        # 转换为张量
        seq_states = torch.stack([torch.FloatTensor(s[0]) for s in states])
        graph_states = torch.stack([torch.FloatTensor(s[1]) for s in states])
        actions = torch.LongTensor(actions)
        old_log_probs = torch.FloatTensor(log_probs)
        rewards = torch.FloatTensor(rewards)
        next_seq = torch.stack([torch.FloatTensor(s[0]) for s in next_states])
        next_graph = torch.stack([torch.FloatTensor(s[1]) for s in next_states])
        dones = torch.FloatTensor(dones)  # 将dones转换为FloatTensor
        
        # 计算目标价值
        with torch.no_grad():
            _, next_values = self.network(next_seq, next_graph)
            targets = rewards + (1 - dones) * self.gamma * next_values.squeeze()
        
        # 计算当前价值
        _, current_values = self.network(seq_states, graph_states)
        advantages = targets - current_values.squeeze()
        
        # 计算Actor损失
        new_probs, _ = self.network(seq_states, graph_states)
        dist = torch.distributions.Categorical(new_probs)
        new_log_probs = dist.log_prob(actions)
        
        ratio = (new_log_probs - old_log_probs).exp()
        actor_loss = -torch.min(
            ratio * advantages,
            torch.clamp(ratio, 0.8, 1.2) * advantages
        ).mean()
        
        # 计算Critic损失
        critic_loss = nn.MSELoss()(current_values.squeeze(), targets)
        
        # 总损失
        total_loss = actor_loss + 0.5 * critic_loss
        
        # 反向传播
        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()

class APTEnv:
    """APT攻击链环境"""
    def __init__(self, sequences):
        self.sequences = sequences
        self.current_seq = None
        self.causal_graph = np.zeros((6, 6))
        
    def reset(self):
        self.current_seq = self.sequences[np.random.choice(len(self.sequences))]
        self.causal_graph = np.zeros((6, 6))
        return self._get_state()
    
    def _get_state(self):
        # 序列编码（滑动窗口）
        window_size = 3
        seq_encoded = np.zeros((window_size, 6))
        for i in range(window_size):
            if i < len(self.current_seq):
                tech = self.current_seq[i]
                seq_encoded[i][tech] = 1
        return seq_encoded, self.causal_graph.copy()
    
    def _causal_reward(self, src, dst):
        # 基于共现概率的因果效应
        count = 0
        total = 0
        for seq in self.sequences:
            if src in seq and dst in seq:
                total += 1
                if seq.index(src) < seq.index(dst):
                    count += 1
        return count / (total + 1e-8)
    
    def step(self, action):
        src, dst = divmod(action, 6)  # 将动作展平为0-35
        
        # 更新因果图
        self.causal_graph[src][dst] = 1
        
        # 计算奖励
        reward = self._causal_reward(src, dst)
        reward -= 0.1 * self.causal_graph.sum()  # 稀疏性惩罚
        
        # 有效性检查
        try:
            valid = self.current_seq.index(src) < self.current_seq.index(dst)
            reward += 0.5 if valid else -0.3
        except ValueError:
            reward -= 0.3
            
        # 终止条件
        done = len(self.current_seq) >= 5
        
        return self._get_state(), reward, done, {}


In [ ]:

# 训练流程
if __name__ == "__main__":
    # 生成模拟数据
    def generate_sequences():
        base_chain = [0, 1, 2, 4]  # 扫描->SQL注入->恶意软件->C2
        return [base_chain.copy() for _ in range(100)]
    
    env = APTEnv(generate_sequences())
    agent = ACAgent(state_dim=6, action_dim=36)  # 6x6=36种可能的边
    
    for episode in range(1000):
        state = env.reset()
        total_reward = 0
        
        while True:
            # 获取动作
            action, log_prob = agent.get_action(state[0], state[1])
            
            # 执行动作
            next_state, reward, done, _ = env.step(action)
            
            # 存储经验
            agent.memory.append((state, action, log_prob, reward, next_state, done))
            
            # 更新网络
            agent.update()
            
            state = next_state
            total_reward += reward
            
            if done:
                break
        
        print(f"Episode {episode}, Reward: {total_reward:.2f}")
    
    # 输出最终因果图
    print("推断的因果关联：")
    for i in range(6):
        for j in range(6):
            if env.causal_graph[i][j] > 0.5:
                print(f"{TECHNIQUES[i]} → {TECHNIQUES[j]}")

In [ ]:
#​因果图精确度
def evaluate_accuracy(pred_graph, true_graph):
    tp = np.sum((pred_graph == 1) & (true_graph == 1))
    fp = np.sum((pred_graph == 1) & (true_graph == 0))
    return tp / (tp + fp + 1e-8)

In [ ]:
#攻击链覆盖率
def coverage_ratio(pred_graph, test_sequences):
    covered = 0
    for seq in test_sequences:
        valid = True
        for i in range(len(seq)-1):
            if pred_graph[seq[i]][seq[i+1]] < 0.5:
                valid = False
                break
        covered += int(valid)
    return covered / len(test_sequences)